In [11]:
# 🎮 MATCH BEÁLLÍTÁSOK
# Melyik játék a sorozatban (1-5)
GAME_INDEX = 3

MATCH_URL = "https://andydanger.github.io/live-lol-esports/#/live/115016265550803595"

ODDS_URL = "https://www.tippmixpro.hu/hu/fogadas/i/esemenyek/100/league-of-legends-lol/vilag/emea-masters-summer/karmine-corp-blue-los-heretics/284726865528393728/palyak"

# 🔄 TEAM MAPPING
HOME_IS_BLUE = True  # True = Hazai csapat BLUE oldalon játszik
                     # False = Hazai csapat RED oldalon játszik

# 🎯 VALUE BET KRITÉRIUMOK
MIN_EDGE = 3.0      # Minimum edge % (pl. 5.0 = 5%)
MIN_CONFIDENCE = 0.00  # Minimum win probability (0.55 = 55%)
USE_ENSEMBLE = True    # Használja mindkét modellt

In [35]:
#🎯 QUICK VALUE BET CHECKER


import sys
import os
from datetime import datetime

# Add hozzá a projekt mappáját a path-hoz (ha szükséges)
# sys.path.append('/path/to/your/project')  # Módosítsd!

from scrapers import scrape_match_stats, scrape_odds
from value_betting import ValueBettingEngine
import joblib

# ============================================================================
# 2️⃣ KONFIGURÁCIÓ - Itt állítsd be a paramétereket
# ============================================================================

# 📁 MODEL PATHS - Egy directoryval feljebb!
MODEL_DIR = os.path.join("..", "models")  # ÚJ: Egy szinttel feljebb!
GB_MODEL_PATH = os.path.join(MODEL_DIR, "live_gb_model_20251031.joblib")
RF_MODEL_PATH = os.path.join(MODEL_DIR, "live_rf_model_20251031.joblib")
SCALER_PATH = os.path.join(MODEL_DIR, "live_scaler_20251031.joblib")
# ============================================================================
# 3️⃣ LOAD MODELS
# ============================================================================

print("📦 Loading models...")
try:
    gb_model = joblib.load(GB_MODEL_PATH)
    rf_model = joblib.load(RF_MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    print("✅ Models loaded successfully!\n")
except Exception as e:
    print(f"❌ Error loading models: {e}")
    print("Make sure model files exist at the specified paths!")
    raise

# ============================================================================
# 4️⃣ MAIN EXECUTION - FUTTASD EZT A CELLÁT!
# ============================================================================

print("="*70)
print(f"🎯 VALUE BET CHECKER - Game {GAME_INDEX}")
print("="*70)
print(f"⏰ Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔄 Team Mapping: {'Hazai → BLUE | Vendég → RED' if HOME_IS_BLUE else 'Hazai → RED | Vendég → BLUE'}")
print(f"📊 Min Edge: {MIN_EDGE}% | Min Confidence: {MIN_CONFIDENCE:.0%}")
print("="*70)
print()

# ============================================================================
# STEP 1: Scrape match stats
# ============================================================================
print("🔍 [1/3] Scraping match stats...")
match_url_with_game = f"{MATCH_URL}/game-index/{GAME_INDEX}"
match_data = scrape_match_stats(match_url_with_game)

if not match_data:
    print("❌ Failed to scrape match data!")
    print("Possible issues:")
    print("  - URL is incorrect")
    print("  - Match is not live")
    print("  - Chrome driver not installed")
    raise Exception("Match data scraping failed")

print(f"✅ Match data scraped!")
print(f"   Game Time: {match_data['game_time']}")
print(f"   Blue: {match_data['blue_team']['kills']} kills, {match_data['blue_team']['gold']:,} gold")
print(f"   Red:  {match_data['red_team']['kills']} kills, {match_data['red_team']['gold']:,} gold")
print()

# ============================================================================
# STEP 2: Scrape odds
# ============================================================================
print("🔍 [2/3] Scraping betting odds...")
odds_data = scrape_odds(ODDS_URL, game_index=GAME_INDEX)

if not odds_data or not odds_data['markets']:
    print("❌ Failed to scrape odds data!")
    print("Possible issues:")
    print("  - URL is incorrect")
    print("  - No markets available for this game")
    print("  - Page structure changed")
    raise Exception("Odds data scraping failed")

print(f"✅ Odds data scraped!")
print(f"   Markets found: {len(odds_data['markets'])}")
for market in odds_data['markets']:
    print(f"   📊 {market['name']}")
    for opt in market['options']:
        print(f"      - {opt['name']}: {opt['odds']:.2f}")
print()

# ============================================================================
# STEP 3: Analyze and find value bets
# ============================================================================
print("🎯 [3/3] Analyzing value betting opportunities...")

engine = ValueBettingEngine(
    gb_model, 
    rf_model, 
    scaler,
    min_edge=MIN_EDGE/100,
    min_confidence=MIN_CONFIDENCE
)

# Calculate features and predictions
features = engine.calculate_features(match_data)
prob_blue, prob_red = engine.predict_win_probability(features, USE_ENSEMBLE)

print(f"🤖 Model Predictions:")
print(f"   🔵 BLUE Team: {prob_blue:.1%} win probability")
print(f"   🔴 RED Team:  {prob_red:.1%} win probability")
print()

# Find value bets
value_bets = engine.find_value_bets(
    match_data, 
    odds_data, 
    USE_ENSEMBLE,
    home_is_blue=HOME_IS_BLUE
)

# ============================================================================
# RESULTS
# ============================================================================
print("="*70)
print("📊 RESULTS")
print("="*70)

if value_bets:
    print(f"✅ Found {len(value_bets)} VALUE BET(S)!\n")
    
    for i, bet in enumerate(value_bets, 1):
        print(f"💰 VALUE BET #{i}")
        print(f"   {'='*60}")
        print(f"   🎲 Team: {bet['team_name']} ({bet['team']} side)")
        print(f"   📊 Market: {bet['market_name']}")
        print(f"   💵 Odds: {bet['odds']:.2f}")
        print(f"   📈 Edge: {bet['edge']:.1f}%")
        print(f"   🎯 Confidence: {bet['confidence']}")
        print(f"   🔮 Predicted Probability: {bet['predicted_prob']:.1%}")
        print(f"   📉 Implied Probability: {bet['implied_prob']:.1%}")
        print(f"   💼 Kelly Stake: {bet['kelly_fraction']:.1%} of bankroll")
        print(f"   ⏰ Game Time: {bet['game_time']}")
        print()
else:
    print("❌ No value bets found at this time.")
    print()
    print("Possible reasons:")
    print(f"  - Model predictions don't exceed odds by {MIN_EDGE}%")
    print(f"  - Win probability is below {MIN_CONFIDENCE:.0%}")
    print("  - Current odds are too accurate")
    print()
    print("💡 Try adjusting MIN_EDGE or MIN_CONFIDENCE settings!")

print("="*70)
print("🏁 Analysis complete!")
print("="*70)

# ============================================================================
# OPTIONAL: Show detailed breakdown
# ============================================================================
print("\n📋 DETAILED BREAKDOWN:")
print(f"   Game Time: {match_data['game_time']}")
print(f"   Blue Score: {match_data['blue_team']['kills']}-{match_data['red_team']['kills']}")
print(f"   Gold Diff: {match_data['blue_team']['gold'] - match_data['red_team']['gold']:+,}")
print(f"   Tower Diff: {match_data['blue_team']['towers'] - match_data['red_team']['towers']:+d}")
print(f"   Dragon Diff: {len(match_data['blue_team']['dragons']) - len(match_data['red_team']['dragons']):+d}")

# Compare odds vs predictions
print("\n🔍 ODDS ANALYSIS:")
for market in odds_data['markets']:
    if market.get('game_index') == GAME_INDEX and 'Ki nyeri' in market['name']:
        for idx, option in enumerate(market['options']):
            is_home = idx == 0
            if HOME_IS_BLUE:
                pred_prob = prob_blue if is_home else prob_red
                team_color = "🔵 BLUE" if is_home else "🔴 RED"
            else:
                pred_prob = prob_red if is_home else prob_blue
                team_color = "🔴 RED" if is_home else "🔵 BLUE"
            
            implied_prob = 1 / option['odds']
            edge = (pred_prob * option['odds'] - 1) * 100
            
            print(f"   {option['name']} ({team_color}):")
            print(f"      Odds: {option['odds']:.2f} | Implied: {implied_prob:.1%} | Predicted: {pred_prob:.1%} | Edge: {edge:+.1f}%")

INFO:scrapers:Starting match stats scraper...


📦 Loading models...
✅ Models loaded successfully!

🎯 VALUE BET CHECKER - Game 3
⏰ Time: 2025-11-02 21:48:01
🔄 Team Mapping: Hazai → BLUE | Vendég → RED
📊 Min Edge: 3.0% | Min Confidence: 0%

🔍 [1/3] Scraping match stats...


INFO:scrapers:✅ Successfully scraped match at 29:02 (Game 3)
INFO:scrapers:Starting odds scraper...


✅ Match data scraped!
   Game Time: 29:02
   Blue: 12 kills, 51,335 gold
   Red:  14 kills, 52,149 gold

🔍 [2/3] Scraping betting odds...


INFO:scrapers:✅ Successfully scraped 1 markets (filtered to game 3)
INFO:value_betting:Model predictions (Game 3) - Blue: 26.1%, Red: 73.9%
INFO:value_betting:Team mapping: Hazai=BLUE, Vendég=RED
INFO:value_betting:🎯 VALUE BET (Game 3): Los Heretics (RED side) at 3.05 (Edge: 125.4%)


✅ Odds data scraped!
   Markets found: 1
   📊 Ki nyeri? - 3. pálya
      - Karmine Corp Blue: 1.33
      - Los Heretics: 3.05

🎯 [3/3] Analyzing value betting opportunities...
🤖 Model Predictions:
   🔵 BLUE Team: 26.1% win probability
   🔴 RED Team:  73.9% win probability

📊 RESULTS
✅ Found 1 VALUE BET(S)!

💰 VALUE BET #1
   🎲 Team: Los Heretics (RED side)
   📊 Market: Ki nyeri? - 3. pálya
   💵 Odds: 3.05
   📈 Edge: 125.4%
   🎯 Confidence: HIGH
   🔮 Predicted Probability: 73.9%
   📉 Implied Probability: 32.8%
   💼 Kelly Stake: 25.0% of bankroll
   ⏰ Game Time: 29:02

🏁 Analysis complete!

📋 DETAILED BREAKDOWN:
   Game Time: 29:02
   Blue Score: 12-14
   Gold Diff: -814
   Tower Diff: +0
   Dragon Diff: +0

🔍 ODDS ANALYSIS:
   Karmine Corp Blue (🔵 BLUE):
      Odds: 1.33 | Implied: 75.2% | Predicted: 26.1% | Edge: -65.3%
   Los Heretics (🔴 RED):
      Odds: 3.05 | Implied: 32.8% | Predicted: 73.9% | Edge: +125.4%


In [36]:
# 🎲 A TÉTED ADATAI
BET_TEAM = "BLUE"  # Melyik csapatra tettél? "BLUE" vagy "RED"
BET_AMOUNT = 100   # Mennyit tettél fel? (Ft)
BET_ODDS = 6    # Milyen oddson? (tizedes formátum)

# 💵 JELENLEGI CASHOUT AJÁNLAT
CASHOUT_AMOUNT = 390  # Mennyit ajánlanak cashout-ra? (Ft)

# ⚙️ DÖNTÉSI PARAMÉTEREK
MIN_PROFIT_TO_CASHOUT = 80  # Minimum profit % amitől megfontolandó a cashout (20%)
MAX_LOSS_TOLERANCE = -100    # Maximum veszteség % aminél automatikusan ki kéne venni (-30%)

In [37]:
# 💰 CASHOUT ANALYZER

# ============================================================================
# 🔍 ELEMZÉS
# ============================================================================

print("="*70)
print("💰 CASHOUT ANALYZER")
print("="*70)
print(f"⏰ Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# Ellenőrizd, hogy van-e már match_data és predictions
if 'match_data' not in globals() or 'prob_blue' not in globals():
    print("⚠️  FIGYELEM: Előbb futtasd le az első cellát a predictions megszerzéséhez!")
    print("   Vagy állítsd be manuálisan:")
    print()
    # Manual override lehetőség
    MANUAL_PROB_BLUE = 0.30  # ← IDE ÍRD BE A JELENLEGI BLUE GYŐZELMI ESÉLYT!
    MANUAL_PROB_RED = 0.70   # ← IDE ÍRD BE A JELENLEGI RED GYŐZELMI ESÉLYT!
    
    #prob_blue = MANUAL_PROB_BLUE
    #prob_red = MANUAL_PROB_RED
    print(f"   📝 Manual predictions használva:")
    print(f"      Blue: {prob_blue:.1%}")
    print(f"      Red: {prob_red:.1%}")
    print()

# Válaszd ki a megfelelő valószínűséget
bet_team_prob = prob_blue if BET_TEAM.upper() == "BLUE" else prob_red

print("📊 TÉTED RÉSZLETEI:")
print(f"   🎲 Csapat: {BET_TEAM}")
print(f"   💵 Tét összege: {BET_AMOUNT:,.0f} Ft")
print(f"   📈 Odds (fogadáskor): {BET_ODDS:.2f}")
print(f"   💰 Potenciális nyeremény: {BET_AMOUNT * BET_ODDS:,.0f} Ft")
print(f"   📉 Potenciális profit: {BET_AMOUNT * (BET_ODDS - 1):,.0f} Ft")
print()

print("🔮 JELENLEGI HELYZET:")
print(f"   🤖 Modell szerinti győzelmi esély: {bet_team_prob:.1%}")
print(f"   💸 Cashout ajánlat: {CASHOUT_AMOUNT:,.0f} Ft")
print()

# ============================================================================
# SZÁMÍTÁSOK
# ============================================================================

# Eredeti fogadáskor:
entry_implied_prob = 1 / BET_ODDS
entry_expected_value = BET_AMOUNT * (bet_team_prob * BET_ODDS - 1)

# Jelenlegi helyzet:
potential_win = BET_AMOUNT * BET_ODDS
current_expected_value = bet_team_prob * potential_win + (1 - bet_team_prob) * 0

# Cashout vs tartás összehasonlítása:
cashout_profit = CASHOUT_AMOUNT - BET_AMOUNT
cashout_profit_pct = (cashout_profit / BET_AMOUNT) * 100

expected_value_if_hold = current_expected_value - BET_AMOUNT
ev_if_hold_pct = (expected_value_if_hold / BET_AMOUNT) * 100

# Döntés:
ev_difference = current_expected_value - CASHOUT_AMOUNT

print("="*70)
print("📊 RÉSZLETES ELEMZÉS")
print("="*70)
print()

print("💡 VÁRHATÓ ÉRTÉKEK:")
print(f"   Ha KIVESZED (Cashout):")
print(f"      ✅ Garantált összeg: {CASHOUT_AMOUNT:,.0f} Ft")
print(f"      📊 Profit/veszteség: {cashout_profit:+,.0f} Ft ({cashout_profit_pct:+.1f}%)")
print()
print(f"   Ha TARTOD a tétet:")
print(f"      🎲 Várható érték: {current_expected_value:,.0f} Ft")
print(f"      📈 Várható profit: {expected_value_if_hold:+,.0f} Ft ({ev_if_hold_pct:+.1f}%)")
print(f"      ✅ Ha nyer: {potential_win:,.0f} Ft (+{potential_win - BET_AMOUNT:,.0f} Ft)")
print(f"      ❌ Ha veszít: 0 Ft (-{BET_AMOUNT:,.0f} Ft)")
print()

print("🎯 VALÓSZÍNŰSÉGEK:")
print(f"   Fogadáskor (implied): {entry_implied_prob:.1%}")
print(f"   Most (modell szerint): {bet_team_prob:.1%}")
print(f"   Változás: {(bet_team_prob - entry_implied_prob):+.1%}")
print()

# ============================================================================
# AJÁNLÁS
# ============================================================================

print("="*70)
print("🎯 AJÁNLÁS")
print("="*70)
print()

recommendation = None
reasons = []

# Döntési logika:
if cashout_profit_pct < MAX_LOSS_TOLERANCE:
    recommendation = "🚨 AZONNAL VEDD KI!"
    reasons.append(f"❌ Nagy veszteség ({cashout_profit_pct:.1f}%) - vágj veszteséget!")
    reasons.append(f"   Jelenleg csak {bet_team_prob:.1%} esély a győzelemre")

elif bet_team_prob < 0.35:
    recommendation = "⚠️  ERŐSEN JAVASOLT KIVENNI"
    reasons.append(f"❌ Túl alacsony győzelmi esély ({bet_team_prob:.1%})")
    reasons.append(f"   A modell szerint kicsi az esély a sikerre")

elif cashout_profit_pct >= MIN_PROFIT_TO_CASHOUT and ev_difference < 0:
    recommendation = "💰 CASHOUT JAVASOLT"
    reasons.append(f"✅ Biztosíts {cashout_profit_pct:.1f}% profitot!")
    reasons.append(f"   A várható érték már nem kompenzálja a kockázatot")
    reasons.append(f"   Különbség: {ev_difference:,.0f} Ft a cashout javára")

elif ev_difference > 100:
    recommendation = "💎 TARTSD A TÉTET!"
    reasons.append(f"✅ A tartás várható értéke magasabb ({ev_difference:+,.0f} Ft)")
    reasons.append(f"   Győzelmi esély: {bet_team_prob:.1%}")
    reasons.append(f"   Várható profit: {expected_value_if_hold:,.0f} Ft")

elif abs(ev_difference) < 100:
    recommendation = "🤔 HATÁRESET"
    reasons.append(f"⚖️  Nagyon kicsi a különbség ({ev_difference:+,.0f} Ft)")
    reasons.append(f"   Matematikailag majdnem mindegy")
    reasons.append(f"   Dönts a kockázatkerülésed alapján:")
    reasons.append(f"      • Konzervatív → Cashout")
    reasons.append(f"      • Kockázatvállaló → Tartsd")

else:
    recommendation = "📊 NEHÉZ DÖNTÉS"
    reasons.append(f"   EV különbség: {ev_difference:+,.0f} Ft")
    reasons.append(f"   Győzelmi esély: {bet_team_prob:.1%}")
    reasons.append(f"   Nézd meg a mérkőzés menetét is!")

print(f"🎲 {recommendation}")
print()
print("Indokok:")
for reason in reasons:
    print(f"   {reason}")

print()
print("="*70)
print("📈 ÖSSZEHASONLÍTÓ TÁBLÁZAT")
print("="*70)
print()
print(f"{'Opció':<20} {'Összeg':<15} {'Profit':<15} {'Esély':<10}")
print("-" * 70)
print(f"{'💸 Cashout':<20} {f'{CASHOUT_AMOUNT:,.0f} Ft':<15} {f'{cashout_profit:+,.0f} Ft':<15} {'100%':<10}")
print(f"{'💎 Tartás (várható)':<20} {f'{current_expected_value:,.0f} Ft':<15} {f'{expected_value_if_hold:+,.0f} Ft':<15} {'—':<10}")
print(f"{'✅ Ha nyer':<20} {f'{potential_win:,.0f} Ft':<15} {f'{potential_win - BET_AMOUNT:+,.0f} Ft':<15} {f'{bet_team_prob:.1%}':<10}")
print(f"{'❌ Ha veszít':<20} {'0 Ft':<15} {f'-{BET_AMOUNT:,.0f} Ft':<15} {f'{1-bet_team_prob:.1%}':<10}")
print()

# ============================================================================
# EXTRA TIP
# ============================================================================
print("💡 EXTRA TIPPEK:")
print()
if bet_team_prob > 0.60:
    print("   ✅ Erős pozíció! Érdemes lehet tartani a tétet.")
elif bet_team_prob < 0.40:
    print("   ⚠️  Gyenge pozíció. Fontold meg a cashout-ot!")
else:
    print("   🤷 Közepesen bizonytalan. Nézd a játék állását is!")

if cashout_profit > 0:
    roi = (cashout_profit / BET_AMOUNT) * 100
    print(f"   💰 Jelenlegi cashout ROI: {roi:.1f}%")
else:
    print(f"   📉 Veszteségben vagy: {cashout_profit_pct:.1f}%")

print()
print("="*70)

💰 CASHOUT ANALYZER
⏰ Time: 2025-11-02 21:48:25

📊 TÉTED RÉSZLETEI:
   🎲 Csapat: BLUE
   💵 Tét összege: 100 Ft
   📈 Odds (fogadáskor): 6.00
   💰 Potenciális nyeremény: 600 Ft
   📉 Potenciális profit: 500 Ft

🔮 JELENLEGI HELYZET:
   🤖 Modell szerinti győzelmi esély: 26.1%
   💸 Cashout ajánlat: 390 Ft

📊 RÉSZLETES ELEMZÉS

💡 VÁRHATÓ ÉRTÉKEK:
   Ha KIVESZED (Cashout):
      ✅ Garantált összeg: 390 Ft
      📊 Profit/veszteség: +290 Ft (+290.0%)

   Ha TARTOD a tétet:
      🎲 Várható érték: 157 Ft
      📈 Várható profit: +57 Ft (+56.6%)
      ✅ Ha nyer: 600 Ft (+500 Ft)
      ❌ Ha veszít: 0 Ft (-100 Ft)

🎯 VALÓSZÍNŰSÉGEK:
   Fogadáskor (implied): 16.7%
   Most (modell szerint): 26.1%
   Változás: +9.4%

🎯 AJÁNLÁS

🎲 ⚠️  ERŐSEN JAVASOLT KIVENNI

Indokok:
   ❌ Túl alacsony győzelmi esély (26.1%)
      A modell szerint kicsi az esély a sikerre

📈 ÖSSZEHASONLÍTÓ TÁBLÁZAT

Opció                Összeg          Profit          Esély     
-------------------------------------------------------------